In [17]:
import os

In [18]:
#uncomment then run once then comment again
#os.chdir('/home/fidel/Documents/Road-accident-blackspot-detection-Project')
#os.getcwd()

In [ ]:
import pandas as pd
import numpy as np
import warnings

# cleaner notebook output
warnings.filterwarnings('ignore')

# Load the dataset
file_path = 'data/raw/ma3route_crashes_algorithmcode.csv'
df = pd.read_csv(file_path)

print(f"Initial dataset shape: {df.shape}")
df.head()

Initial dataset shape: (31064, 10)


,crash_id,crash_datetime,crash_date,latitude,longitude,n_crash_reports,contains_fatality_words,contains_pedestrian_words,contains_matatu_words,contains_motorcycle_words
0,1,2018-06-06 20:39:54,2018-06-06,-1.263030,36.764374,1,0,0,0,0
1,2,2018-08-17 06:15:54,2018-08-17,-0.829710,37.037820,1,1,0,0,0
2,3,2018-05-25 17:51:54,2018-05-25,-1.125301,37.003297,1,0,0,0,0
3,4,2018-05-25 18:11:54,2018-05-25,-1.740958,37.129026,1,0,0,0,0
4,5,2018-05-25 21:59:54,2018-05-25,-1.259392,36.842321,1,1,0,0,0


REMOVING DUPLICATES


In [ ]:
#  Removing duplicate rows
df = df.drop_duplicates()

# Removing duplicates based on crash_id (keeping the first occurrence)
df = df.drop_duplicates(subset=['crash_id'], keep='first')

print(f"Shape after removing duplicates: {df.shape}")

Shape after removing duplicates: (31064, 10)


HANDLING MISSING COORDINATES

In [ ]:
# Check for missing coordinates before cleaning
missing_coords = df['latitude'].isna().sum() + df['longitude'].isna().sum()
print(f"Rows with missing coordinates before cleaning: {missing_coords}")

# Dropping rows where either latitude or longitude is missing
df = df.dropna(subset=['latitude', 'longitude'])

# Ensure coordinates are numeric
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# Dropping any rows that became NaN after coercion
df = df.dropna(subset=['latitude', 'longitude'])

print(f"Shape after handling missing coordinates: {df.shape}")

Rows with missing coordinates before cleaning: 0
Shape after handling missing coordinates: (31064, 10)


**Standardizing Dates, Times, and Severity Indicators**

In [22]:
# 1. Standardizing dates and times
df['crash_datetime'] = pd.to_datetime(df['crash_datetime'], errors='coerce')
df['crash_date'] = pd.to_datetime(df['crash_date'], errors='coerce').dt.date

# 2. Standardize severity indicators (ensure they are strictly 0 or 1)
severity_cols = [
    'contains_fatality_words', 
    'contains_pedestrian_words', 
    'contains_matatu_words', 
    'contains_motorcycle_words'
]

for col in severity_cols:
    # Convert to numeric, coerce errors to NaN, fill NaN with 0, and ensure integer type
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    # Clip values to ensure they are strictly 0 or 1
    df[col] = df[col].clip(0, 1)

print("Dates and severity indicators standardized.")

Dates and severity indicators standardized.


ADDITION OF USEFUL VARIABLES

In [ ]:
#  Hour of accident
df['hour_of_accident'] = df['crash_datetime'].dt.hour

# Day of week (0=Monday, 6=Sunday)
df['day_of_week_num'] = df['crash_datetime'].dt.dayofweek
df['day_of_week_name'] = df['crash_datetime'].dt.day_name()

#  Month
df['month'] = df['crash_datetime'].dt.month
df['month_name'] = df['crash_datetime'].dt.month_name()

#  Weekend vs Weekday (True if Saturday or Sunday)
df['is_weekend'] = df['day_of_week_num'].isin([5, 6])

# Explicit Involvement Indicators (Boolean)
df['fatality_indicator'] = df['contains_fatality_words'].astype(bool)
df['motorcycle_involvement'] = df['contains_motorcycle_words'].astype(bool)
df['pedestrian_involvement'] = df['contains_pedestrian_words'].astype(bool)

# Severity Score (Weighted)
# Fatality = 3, Pedestrian = 2, Matatu = 1, Motorcycle = 1
#  adding  a scaled version of n_crash_reports (capped at 3 to prevent skewing), helping to differentiate between single and multiple crash locations
df['severity_score'] = (
    (df['contains_fatality_words'] * 3) +
    (df['contains_pedestrian_words'] * 2) +
    (df['contains_matatu_words'] * 1) +
    (df['contains_motorcycle_words'] * 1) +
    (df['n_crash_reports'].clip(upper=3))
)

print("Useful variables created successfully.")
df[['crash_datetime', 'hour_of_accident', 'day_of_week_name', 'is_weekend', 'severity_score']].head()

Useful variables created successfully.


,crash_datetime,hour_of_accident,day_of_week_name,is_weekend,severity_score
0,2018-06-06 20:39:54,20,Wednesday,False,1
1,2018-08-17 06:15:54,6,Friday,False,4
2,2018-05-25 17:51:54,17,Friday,False,1
3,2018-05-25 18:11:54,18,Friday,False,1
4,2018-05-25 21:59:54,21,Friday,False,4


SEGMENTATION OF THE ROADS

In [ ]:
# Defining grid size in degrees (~500 meters), confirm my friends
# degree of latitude ≈ 111 km. Therefore, 0.0045 degrees ≈ 500 meters.
grid_size = 0.0045

# Calculate the grid cell coordinates for each crash
df['segment_lat'] = (df['latitude'] // grid_size) * grid_size
df['segment_lon'] = (df['longitude'] // grid_size) * grid_size

# Create a unique Segment ID by combining the rounded lat and lon
# Rounding to 4 decimal places for cleaner segment names
df['road_segment_id'] = 'SEG_' + df['segment_lat'].round(4).astype(str) + '_' + df['segment_lon'].round(4).astype(str)

print("Road segments assigned successfully.")
print(f"Total unique road segments created: {df['road_segment_id'].nunique()}")
df[['latitude', 'longitude', 'segment_lat', 'segment_lon', 'road_segment_id']].head()

Road segments assigned successfully.
Total unique road segments created: 1693


,latitude,longitude,segment_lat,segment_lon,road_segment_id
0,-1.263030,36.764374,-1.2645,36.7605,SEG_-1.2645_36.7605
1,-0.829710,37.037820,-0.8325,37.0350,SEG_-0.8325_37.035
2,-1.125301,37.003297,-1.1295,36.9990,SEG_-1.1295_36.999
3,-1.740958,37.129026,-1.7415,37.1250,SEG_-1.7415_37.125
4,-1.259392,36.842321,-1.2600,36.8415,SEG_-1.26_36.8415


FINAL REVIEW

In [25]:
import os

# ==========================================
# CONFIGURE YOUR FOLDER PATH HERE
# ==========================================
# Replace 'data' with the actual folder name where your datasets are kept.
# Examples: 'datasets', 'data/cleaned', '../data'
dataset_folder = 'data/raw' 

# Create the folder if it doesn't already exist (prevents errors)
os.makedirs(dataset_folder, exist_ok=True)

# Define the final column order for clarity
final_columns = [
    'crash_id', 'crash_datetime', 'crash_date', 'latitude', 'longitude',
    'road_segment_id', 'n_crash_reports', 'severity_score',
    'fatality_indicator', 'pedestrian_involvement', 'motorcycle_involvement',
    'hour_of_accident', 'day_of_week_name', 'is_weekend', 'month_name',
    'contains_fatality_words', 'contains_pedestrian_words', 
    'contains_matatu_words', 'contains_motorcycle_words'
]

# Ensure all created columns exist before reordering
existing_columns = [col for col in final_columns if col in df.columns]
df_cleaned = df[existing_columns].copy()

# Display final info
print("=== Final Dataset Info ===")
print(df_cleaned.info())
print("\n=== Sample of Final Dataset ===")
display(df_cleaned.head())

# Construct the full file path using the folder and filename
output_file = os.path.join(dataset_folder, 'ma3route_crashes_cleaned.csv')

# Export to the specified directory
df_cleaned.to_csv(output_file, index=False)
print(f"\n✅ Cleaned dataset successfully saved to: {output_file}")

=== Final Dataset Info ===
<class 'pandas.DataFrame'>
RangeIndex: 31064 entries, 0 to 31063
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   crash_id                   31064 non-null  int64         
 1   crash_datetime             31064 non-null  datetime64[us]
 2   crash_date                 31064 non-null  object        
 3   latitude                   31064 non-null  float64       
 4   longitude                  31064 non-null  float64       
 5   road_segment_id            31064 non-null  str           
 6   n_crash_reports            31064 non-null  int64         
 7   severity_score             31064 non-null  int64         
 8   fatality_indicator         31064 non-null  bool          
 9   pedestrian_involvement     31064 non-null  bool          
 10  motorcycle_involvement     31064 non-null  bool          
 11  hour_of_accident           31064 non-null  int32   

,crash_id,crash_datetime,crash_date,latitude,longitude,road_segment_id,n_crash_reports,severity_score,fatality_indicator,pedestrian_involvement,motorcycle_involvement,hour_of_accident,day_of_week_name,is_weekend,month_name,contains_fatality_words,contains_pedestrian_words,contains_matatu_words,contains_motorcycle_words
0,1,2018-06-06 20:39:54,2018-06-06,-1.263030,36.764374,SEG_-1.2645_36.7605,1,1,False,False,False,20,Wednesday,False,June,0,0,0,0
1,2,2018-08-17 06:15:54,2018-08-17,-0.829710,37.037820,SEG_-0.8325_37.035,1,4,True,False,False,6,Friday,False,August,1,0,0,0
2,3,2018-05-25 17:51:54,2018-05-25,-1.125301,37.003297,SEG_-1.1295_36.999,1,1,False,False,False,17,Friday,False,May,0,0,0,0
3,4,2018-05-25 18:11:54,2018-05-25,-1.740958,37.129026,SEG_-1.7415_37.125,1,1,False,False,False,18,Friday,False,May,0,0,0,0
4,5,2018-05-25 21:59:54,2018-05-25,-1.259392,36.842321,SEG_-1.26_36.8415,1,4,True,False,False,21,Friday,False,May,1,0,0,0



✅ Cleaned dataset successfully saved to: data/raw/ma3route_crashes_cleaned.csv
